In [1]:
import pandas as pd

In [2]:
df_train = pd.read_csv('D:\\repos\\ML_innovise\\notebooks\\df_train_eda.csv')
df_test  = pd.read_csv('D:\\repos\\ML_innovise\\notebooks\\df_test_eda.csv')

In [3]:
from Splitter import Splitter

splitter = Splitter(time_col = 'date_block_num', train_end = 33)

In [5]:
df_train.head()

,shop_id,item_id,item_category_id,simple_category_id,date_block_num,item_cnt_month,item_price,item_avg_price_prev,avg_price_prev_missing,cnt_lag_1,cnt_lag_2,cnt_lag_3,cnt_lag_12,cnt_rm_3,cnt_rm_6
0,2,30,40,7,0,0.0,0.0,199.0,1,0.0,0.0,0.0,0.0,0.000000,0.000000
1,2,30,40,7,1,0.0,0.0,199.0,1,0.0,0.0,0.0,0.0,0.000000,0.000000
2,2,30,40,7,2,1.0,359.0,199.0,1,0.0,0.0,0.0,0.0,0.000000,0.000000
3,2,30,40,7,3,0.0,0.0,359.0,0,1.0,0.0,0.0,0.0,0.333333,0.333333
4,2,30,40,7,4,0.0,0.0,199.0,1,0.0,1.0,0.0,0.0,0.333333,0.250000


In [6]:
from Validator import Validator
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from math import sqrt

validator = Validator(df_train, splitter, model = Ridge( alpha = 1.0), metric   = lambda y, yhat: sqrt(mean_squared_error(y, yhat)))
features =['cnt_lag_1','cnt_lag_3','cnt_lag_2','cnt_lag_12', 'cnt_rm_3','cnt_rm_6','item_avg_price_prev', 'item_price','date_block_num','avg_price_prev_missing']
result = validator.run(features, target_col = 'item_cnt_month')

print('Val RMSE:', result)

Val RMSE: {'val_score': 0.9030601736450019, 'pred_val': array([0.05930102, 0.23568534, 0.25194839, ..., 0.19877994, 0.05413232,
       0.05413232], shape=(214200,))}


In [7]:
df_testt = pd.read_csv(r'D:\ML_innovise\test.csv')
df_test = df_test.merge(
    df_testt,
    on = ['shop_id','item_id'],
    how = 'left'
)

In [9]:
X_tr, y_tr, X_val, y_val = splitter.split(
    df_train,
    features,            
    target_col="item_cnt_month"
)
X_full = pd.concat([X_tr, X_val], axis=0)
y_full = pd.concat([y_tr, y_val], axis=0)

final_model = Ridge(alpha=1.0, random_state=42)
final_model.fit(X_full, y_full)


X_test = df_test[features]


test_pred = final_model.predict(X_test)


submission = pd.DataFrame({
    "ID":              df_test["ID"],
    "item_cnt_month":  test_pred
})

submission.to_csv("submission2.csv", index=False)